In [1]:
# ==============================================================================
# NOTEBOOK: Model-Training-Milestone3-Refined.ipynb
# DESCRIPTION: Training with Validation Split, Smart Baseline & Pipeline Safety
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Set global random state
RANDOM_STATE = 42

# ===========================================================
# STAGE 1: LOAD AND ALIGN DATA
# ===========================================================
print("="*60)
print("STAGE 1: LOADING & ALIGNING DATA")
print("="*60)

batsman = pd.read_csv('batsman_features_final.csv')
batsman['date'] = pd.to_datetime(batsman['date'])

# Define Target
y = batsman['runs']

# Define Features (Drop leakage & identifiers)
drop_cols = [
    'runs', 'balls_faced', 'strike_rate', 'boundaries', 'got_out', 
    'target_runs', 'target_balls_faced', 'target_strike_rate',
    'match_id', 'player', 'date', 'next_opponent', 'next_venue'
]
X = batsman.drop(columns=drop_cols)

print(f"Features Shape: {X.shape}")



STAGE 1: LOADING & ALIGNING DATA
Features Shape: (14805, 44)


In [2]:
# ===========================================================
# STAGE 2: TRAIN - VALIDATION - TEST SPLIT (Strict Time Series)
# ===========================================================
print("\n" + "="*60)
print("STAGE 2: TIME-SERIES SPLIT (TRAIN / VAL / TEST)")
print("="*60)

# Split 1: Test Set is strictly 2024 onwards
test_mask = batsman['date'] >= '2024-01-01'
X_test = X[test_mask].drop(columns=['season'])
y_test = y[test_mask]

# Split 2: Validation Set is the 2023 Season (for XGBoost early stopping)
# Everything before 2023 is Training
val_mask = (batsman['date'] >= '2023-01-01') & (batsman['date'] < '2024-01-01')
train_mask = batsman['date'] < '2023-01-01'

X_val = X[val_mask].drop(columns=['season'])
y_val = y[val_mask]

X_train = X[train_mask].drop(columns=['season'])
y_train = y[train_mask]

print(f"Training Set   (2008-2022): {X_train.shape}")
print(f"Validation Set (2023 Only): {X_val.shape}")
print(f"Testing Set    (2024-2025): {X_test.shape}")

# ===========================================================
# STAGE 3: DEFINE PREPROCESSING PIPELINE
# ===========================================================
cat_cols = ['team', 'opponent', 'venue', 'city', 'innings', 'experience_level']
num_cols = [col for col in X_train.columns if col not in cat_cols]

# Preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)




STAGE 2: TIME-SERIES SPLIT (TRAIN / VAL / TEST)
Training Set   (2008-2022): (12105, 43)
Validation Set (2023 Only): (954, 43)
Testing Set    (2024-2025): (1746, 43)


In [3]:
# ===========================================================
# STAGE 4: EVALUATE BASELINES (SMART VS DUMB)
# ===========================================================
print("\n" + "="*60)
print("STAGE 4: BASELINE COMPARISON")
print("="*60)

# 1. Dumb Baseline (Global Mean)
mean_pred = np.full(len(y_test), y_train.mean())
mae_mean = mean_absolute_error(y_test, mean_pred)

# 2. Smart Baseline (Recent Form: runs_last_10)
# We use the raw feature 'runs_last_10' from the test set as the prediction
smart_pred = X_test['runs_last_10'].fillna(y_train.mean()) # Fill NaNs with mean just in case
mae_smart = mean_absolute_error(y_test, smart_pred)

print(f"📉 Dumb Baseline (Global Mean): MAE = {mae_mean:.2f}")
print(f"📉 Smart Baseline (Last 10 Avg): MAE = {mae_smart:.2f}")
print(f"   (If our model can't beat {mae_smart:.2f}, it's useless!)")




STAGE 4: BASELINE COMPARISON
📉 Dumb Baseline (Global Mean): MAE = 18.16
📉 Smart Baseline (Last 10 Avg): MAE = 17.56
   (If our model can't beat 17.56, it's useless!)


In [4]:
# ===========================================================
# STAGE 5: TRAIN & EVALUATE MODELS (WITH PIPELINE SAFETY)
# ===========================================================
print("\n" + "="*60)
print("STAGE 5: MODEL TRAINING")
print("="*60)

# --- Random Forest Pipeline ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE
    ))
])

print("🌲 Training Random Forest Pipeline...")
rf_pipeline.fit(X_train, y_train) # Pipeline handles preprocessing automatically!
y_pred_rf = rf_pipeline.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"✅ Random Forest MAE: {mae_rf:.2f}")

# --- XGBoost (Needs manual handling for Eval Set) ---
print("\n🚀 Training XGBoost (with proper Validation set)...")

# We must preprocess X_val and X_train manually for XGBoost's eval_set
# (XGBoost .fit() doesn't play nicely with Pipelines for eval_set data)
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.01, max_depth=5, 
    early_stopping_rounds=50, n_jobs=-1, random_state=RANDOM_STATE
)

# Fit using Train for learning, Validation for stopping (Test is hidden!)
xgb_model.fit(
    X_train_proc, y_train,
    eval_set=[(X_val_proc, y_val)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test_proc)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
print(f"✅ XGBoost MAE: {mae_xgb:.2f}")




STAGE 5: MODEL TRAINING
🌲 Training Random Forest Pipeline...
✅ Random Forest MAE: 10.48

🚀 Training XGBoost (with proper Validation set)...
✅ XGBoost MAE: 11.06


In [5]:
# ===========================================================
# STAGE 6: FINAL SAVE (ROBUST)
# ===========================================================
print("\n" + "="*60)
print("STAGE 6: SAVING FINAL ARTIFACTS")
print("="*60)

# Save the Pipeline (Single file! Safest for deployment)
joblib.dump(rf_pipeline, 'batsman_pipeline.pkl')
print("✅ Saved: batsman_pipeline.pkl (Contains Preprocessor + Model)")

# Save XGBoost separately (since it's not in the sklearn pipeline object)
joblib.dump(xgb_model, 'batsman_xgb.pkl')
joblib.dump(preprocessor, 'batsman_preprocessor.pkl') 
print("✅ Saved: batsman_xgb.pkl & batsman_preprocessor.pkl")

# Comparison
results_df = pd.DataFrame({
    'Model': ['Mean Baseline', 'Smart Baseline (Form)', 'Random Forest', 'XGBoost'],
    'MAE': [mae_mean, mae_smart, mae_rf, mae_xgb]
})
print("\nFinal Leaderboard:")
print(results_df.sort_values('MAE'))


STAGE 6: SAVING FINAL ARTIFACTS
✅ Saved: batsman_pipeline.pkl (Contains Preprocessor + Model)
✅ Saved: batsman_xgb.pkl & batsman_preprocessor.pkl

Final Leaderboard:
                   Model        MAE
2          Random Forest  10.476911
3                XGBoost  11.060907
1  Smart Baseline (Form)  17.555162
0          Mean Baseline  18.155158
